# Demo 4jun25

## Programmatic methods to sample & compare NS and SRNS points

In [ ]:
# Custom modules with their classes inside
import behaviors
import code_legacy.extract_hyperplanes as extract_hyperplanes  # Unfinished module
import no_signaling_sets
import numpy as np


## Experiment parameters

In [4]:
delta = 2 # Number of outputs a,b
m = 2     # Number of inputs x,y

## Get a non-SRNS point from database

In [5]:
data = np.load("../data/non_srns/non_srns_points.npy")
example_point = data[np.random.randint(len(data))]

print("Example point as vector:", example_point)
print("\n\n")
print("Example point as behavior:")
behavior = behaviors.RoutedBehavior(delta, m, example_point)
print(behavior)

Example point as vector: [0.42298087 0.17582013 0.17351258 0.18063904 0.07340962 0.32057036
 0.33474004 0.32761358 0.1933811  0.19119388 0.44284939 0.18637497
 0.31022841 0.31241563 0.04889799 0.30537242 0.44850901 0.07723225
 0.21808236 0.03058044 0.04788148 0.41915824 0.29017026 0.47767217
 0.19533618 0.31077094 0.42576283 0.35742274 0.30827332 0.19283857
 0.06598455 0.13432464]



Example point as behavior:
Behavior:
Short path (z=S):
[[0.42298087 0.17582013 0.17351258 0.18063904]
 [0.07340962 0.32057036 0.33474004 0.32761358]
 [0.1933811  0.19119388 0.44284939 0.18637497]
 [0.31022841 0.31241563 0.04889799 0.30537242]]
Long path (z=L) :
[[0.44850901 0.07723225 0.21808236 0.03058044]
 [0.04788148 0.41915824 0.29017026 0.47767217]
 [0.19533618 0.31077094 0.42576283 0.35742274]
 [0.30827332 0.19283857 0.06598455 0.13432464]]
------------


### Elementary tests on behaviors

In [6]:
print(f"Coordinates are positive                      : {behavior.positivity()}")
print(f"Coordinates are normalized                    : {behavior.normalization()}")
print(f"Coordinates verify the no-signaling conditions: {behavior.no_signaling()}")
print()
print("Aggregated tests (checks all previous tests):")
print(f"  Normalization                               : {behavior.is_normalized()}")
print(f"  No-signaling                                : {behavior.is_no_signaling()}")

Coordinates are positive                      : True
Coordinates are normalized                    : True
Coordinates verify the no-signaling conditions: True

Aggregated tests (checks all previous tests):
  Normalization                               : True
  No-signaling                                : True


## Instanciate a set to test for belonging in SRNS

In [7]:
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)
srns_set

### Pipelined belonging test

In [8]:
print(f"Does SRNS set contains the example point: {srns_set.is_in_set(behavior)}")

Does SRNS set contains the example point: False


### Main steps to belonging test

#### [1] Get the equation to test for belonging in matrix form

In [9]:
A,b = srns_set.get_equations(behavior)

with np.printoptions(threshold=np.inf, linewidth=np.inf, precision=2): # type: ignore
    print("A matrix:")
    print(A[:-4])
    print("Last 4 rows of A matrix")
    print(A[-4:])
    print()
    print("b vector:")
    print(b)

A matrix:
[[-0.17  1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.07  0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.08  0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.07  0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.18  0.    0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0. 

#### [2] Solve the linear program

In [18]:
from scipy.optimize import OptimizeResult

result: OptimizeResult = srns_set.lp_test(behavior)

In [19]:
print(f"Alpha value for  alpha*p + (1-alpha)*I  : {-result.fun}", "< 1" if -result.fun < 1 else ">= 1")  # noqa: E501

print()

latent_q_vector = result.x[1:]
latent_behavior = behaviors.LatentSRNSBehavior(delta, m, latent_q_vector)
print("Latent behavior:")
print(latent_behavior)
print("Latent behavior is no-signaling: ", latent_behavior.no_signaling())

Alpha value for  alpha*p + (1-alpha)*I  : 0.963632953789202 < 1

Latent behavior:
Behavior:
Short path (z=S):
[[0.41669007 0.17851783 0.1762942  0.18316149]
 [0.07983169 0.31800393 0.33165829 0.324791  ]
 [0.19544016 0.19333248 0.43583603 0.18868882]
 [0.30803808 0.31014576 0.05621148 0.30335868]]
Long path (z=L) :
[[0.0835153  0.        ]
 [0.35777452 0.21924311]
 [0.         0.03856009]
 [0.05523194 0.2501493 ]
 [0.19732414 0.28083945]
 [0.         0.13853141]
 [0.11123673 0.07267665]
 [0.19491736 0.        ]]
------------
Latent behavior is no-signaling:  True


In [12]:
def format_lambda_to_hyperplane(lam: np.ndarray) -> str:
    return str(lam[:16]) + str(lam[16:32]) + str(lam[32:]) 

In [13]:
_, _, lambda_var = srns_set.is_facet_hyperplane(behavior)
hyperplanes_extractor = extract_hyperplanes.HyperplanesExtractor(delta, m, list(data))
print("Corresponding lambda variable:")
with np.printoptions(threshold=np.inf, precision=2):  # type: ignore
    print(lambda_var)
print(len(lambda_var), "is the number of coordinates in the dual variable")
print()

hyperplane = hyperplanes_extractor.scale_down_vector(lambda_var)
print("Rescaled and sliced to the hyperplane size:")
print(format_lambda_to_hyperplane(hyperplane))

Corresponding lambda variable:
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    1.93  0.    0.    0.    1.93 -1.93
  0.    0.    0.    0.   -1.93  1.93  1.93  0.    0.    0.    1.93  0.  ]
36 is the number of coordinates in the dual variable

Rescaled and sliced to the hyperplane size:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0][ 0  0  1  0  0  0  1 -1  0  0  0  0 -1  1  1  0][0 0 1 0]


In [ ]:
np.dot(lambda_var[:32], behavior.get_vector())


np.float64(-0.03636704621079767)

In [28]:
np.dot(lambda_var[:32], behaviors.completely_mixed_behavior.get_vector())

np.float64(0.9636329537892023)

# Sampling

In [14]:
import samplers

In [15]:
sampler = samplers.NoSignalingSampler(delta, m, True)

In [16]:
sampled_vec = sampler.sample()
print("Sampled vector:")
print(sampled_vec)

2025-06-04 11:23:05.459 | SUCCESS  | samplers:sample_multiple:116 - Samples shape: (1, 32)


Sampled vector:
Behavior:
Short path (z=S):
[[0.0281559  0.12497484 0.28957338 0.1237525 ]
 [0.27348693 0.17666798 0.39036603 0.55618691]
 [0.45613139 0.2891345  0.19471391 0.29035684]
 [0.24222578 0.40922267 0.12534668 0.02970375]]
Long path (z=L) :
[[0.19192675 0.21059841 0.26396126 0.03677641]
 [0.10971608 0.09104442 0.41597815 0.643163  ]
 [0.09038782 0.08883315 0.0183533  0.26265515]
 [0.60796936 0.60952402 0.30170729 0.05740544]]
------------


In [17]:
srns_set

In [ ]:
identity = behaviors.completely_mixed_behavior


Behavior([0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25 0.25 0.25])

In [23]:
result = srns_set.lp_test(identity)

In [25]:
-result.fun

2.0

# Post-demo : testing on quantum NS distributions

In [ ]:
from qutip import Qobj, basis, expect, ket2dm, qeye, sigmax, sigmaz, tensor


def projectors(op: Qobj):
    """Return projectors Π_{+1} and Π_{-1} for a Hermitian observable with eigenvalues ±1"""
    eigvals, eigvecs = op.eigenstates()
    proj_dict = {}
    for val, vec in zip(eigvals, eigvecs):
        key = int(np.sign(val)) if val != 0 else +1  # Disambiguate 0 as +1
        proj_dict[key] = ket2dm(vec)
    return proj_dict[+1], proj_dict[-1]

def bell_conditional_distribution(
    alice_ops: list[Qobj],
    bob_ops: list[Qobj]
) -> dict[tuple[int, int], dict[tuple[int, int], float]]:
    """Compute P(a, b | x, y) for all a,b ∈ {±1}, x in Alice ops, y in Bob ops"""

    # Create Φ⁺ state
    ket0 = basis(2, 0)
    ket1 = basis(2, 1)
    phi_plus = (tensor(ket0, ket0) + tensor(ket1, ket1)).unit()
    rho = ket2dm(phi_plus)

    outcomes = [-1, +1]
    distribution = dict()

    for x_index, Ax in enumerate(alice_ops):
        for y_index, By in enumerate(bob_ops):
            # Projectors Π^A_a ⊗ Π^B_b
            Pa_pos, Pa_neg = projectors(Ax)
            Pb_pos, Pb_neg = projectors(By)

            Pxy = dict()
            for a in outcomes:
                Pa = Pa_pos if a == +1 else Pa_neg
                for b in outcomes:
                    Pb = Pb_pos if b == +1 else Pb_neg
                    M = tensor(Pa, Pb)
                    prob = (M * rho).tr().real
                    Pxy[(a, b)] = round(prob, 10)  # Clean rounding
            distribution[(x_index, y_index)] = Pxy

    return distribution
